STEP 8: SIMILARITY COMPUTATION -> RECOMMENDATION ENGINE (USING  COSINE SIMILARITY)

In [3]:
# load  thee saved features(embeddings) for retrieving embeddings of images
# load the saved paths i.e image paths containing actual location of all images (women images  folder)
# load df_images csv to retrieve the metadata  of the selected image


#import pandas as pd
#import pickle

#features = pickle.load(open("features_embeddings.pkl", "rb"))
#paths = pickle.load(open("paths.pkl", "rb"))

#df_images = pd.read_csv("data\\image_dataset_for_modelling.csv")

# check all are of same count and size
#print(len(features), len(paths), len(df_images))

In [4]:
import numpy as np
import pickle
import pandas as pd
import os

from tensorflow.keras.models import load_model, Model
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.efficientnet import preprocess_input

from sklearn.metrics.pairwise import cosine_similarity

In [5]:
# Load model
model = load_model("efficientnet_model.keras")

# Convert to feature extractor
feature_extractor = Model(
    inputs=model.input,
    outputs=model.layers[-2].output
)

# Load embeddings + paths
features = pickle.load(open("features_embeddings.pkl", "rb"))
paths = pickle.load(open("paths.pkl", "rb"))

# Load dataframe (metadata)
df_images = pd.read_csv("data\\image_dataset_for_modelling.csv")

In [6]:
def extract_features(img_path, model):

    img = image.load_img(img_path, target_size=(224,224))
    img = image.img_to_array(img)

    img = np.expand_dims(img, axis=0)
    img = preprocess_input(img)

    features = model.predict(img, verbose=0)
    return features.flatten()

In [7]:
def recommend(img_path, top_n=5):

    # Step 1: extract query features
    query_features = extract_features(img_path, feature_extractor)

    # Step 2: compute similarity
    similarities = cosine_similarity([query_features], features)[0]

    # Step 3: get top matches
    indices = np.argsort(similarities)[::-1]

    # remove the same image if exists
    indices = indices[1:top_n+1]

    results = []

    for i in indices:

        img_path_result = paths[i]
        filename = os.path.basename(img_path_result)

        # SAFE mapping using filename
        metadata = df_images[df_images['filename'] == filename]

        results.append({
            "image_path": img_path_result,
            "similarity": similarities[i],
            "metadata": metadata
        })

    return results

In [8]:
query_image = r"D:\PROJECTS\FASHION PRODUCT\notebooks\women_images\10007.jpg"

results = recommend(query_image, top_n=5)

for i, res in enumerate(results):
    print(f"\nResult {i+1}")
    print("Path:", res["image_path"])
    print("Similarity:", res["similarity"])
    print(res["metadata"])


Result 1
Path: D:\PROJECTS\FASHION PRODUCT\notebooks\women_images\38587.jpg
Similarity: 0.9813831
         id          product_display_name brand_name  price  discounted_price  \
5152  38587  Nike Women White Jdi T-Shirt       Nike  895.0             895.0   

      myntra_rating     age_group gender master_category sub_category  \
5152              1  Adults-Women  Women         Apparel      Topwear   

     article_type base_colour   usage  season    year display_categories  \
5152      Tshirts       White  Casual  Summer  2012.0        Casual Wear   

                                              image_url   filename  \
5152  http://assets.myntassets.com/v1/images/style/p...  38587.jpg   

                                                   link  
5152  http://assets.myntassets.com/v1/images/style/p...  

Result 2
Path: D:\PROJECTS\FASHION PRODUCT\notebooks\women_images\38595.jpg
Similarity: 0.9733764
         id                       product_display_name brand_name  price  \
5160  